*NB: Denne notebooken lar deg arbeide programmatisk med Nasjonalbiblioteket samling av nettavis-tekster (2019-22):*
- *bygge og visualisere korpus*
- *hente konkordanser,*
- *beregne relativfrekvens for kollokasjoner.*

*Har du spørsmål? Les gjerne [overordnet informasjon om nettavis-korpuset](https://www.nb.no/samlingen/nettarkivet/forskning/korpus-med-nettaviser/) eller send oss en e-post på [nettarkivet@nb.no](mailto:nettarkivet@nb.no).*

___

# 0. Importer `dhlab` for python

Før vi begynner må vi importere nødvendige python-pakker. Dersom du ikke har installert `dhlab` allerede må du avkommentere den første linja og fjerne `# ` foran `!pip install ...`

In [1]:
# !pip install -U dhlab (fjern `# ` for å installere siste versjon av python-pakka for dhlab
import dhlab as dh
# import dhlab.nbtext as nb
from dhlab import Corpus, totals, Collocations, Ngram
import plotly.express as px
import numpy as np
import pandas as pd

___

# 1. Korpusanalyse

Et korpus betyr en samling av tekster. I denne sammenhengen skal vi lage et korpus med tekster fra norske nettaviser.

I `dhlab` brukes Python-klassen `Corpus` til å representere en liste med metadata om hvert dokument. Her finner du metadata om publikasjonstittel, språk, dato for innhøsting, domenenavn, osv. Hver tekst i korpuset har også en unik `dhlabid`, DHlabs persistente URN.

Hvis du ønsker en oversikt over tekstenes ulike attributter, slik de eksponeres gjennom APIet, kan du lese mer om [Korpus med nettaviser](https://www.nb.no/samlingen/nettarkivet/forskning/korpus-med-nettaviser/#hvilke-skjema-attributter-kan-jeg-benytte-mot-apiet?).

## 1.1 Bygge korpus

La oss bygge et korpus!

Kodecellen under bygger et korpus med tekster fra NRK som inneholder ord som begynner på "vindmølle*".

`doctype="nettavis"` angir at det er nettavis-tekst vi vil jobbe med. `title="NRK"` avgrenser til tekster fra NRK, og `fulltext="vindmølle*"` avgrenser til tekster som inneholder ord som begynner på vindmølle.

In [14]:
corpus = dh.Corpus(doctype="nettavis", fulltext="ungdom OR ungdomm*", lang="nob OR nno", limit=100000)
corpus

,dhlabid,urn,title,authors,oaiid,sesamid,isbn10,city,timestamp,year,publisher,langs,subjects,ddc,genres,literaryform,doctype
0,601765779,<urn:uuid:e7917e99-ffcf-4b25-90b3-76d24eae95d5>,VG,None,https://www.vg.no:443/nyheter/innenriks/i/4qME...,None,None,Oslo,20200903,2020,vg.no,nob,None,None,None,None,nettavis
1,601775195,<urn:uuid:3056fd32-48f4-43d3-8d37-b62eb3c85a47>,VG,None,https://www.vg.no:443/nyheter/innenriks/i/x320...,None,None,Oslo,20210412,2021,vg.no,nob,None,None,None,None,nettavis
2,601772275,<urn:uuid:c45ae98f-1288-4970-a350-bd96d8ee8e95>,VG,None,https://www.vg.no:443/sport/fotball/i/P9GdPe/c...,None,None,Oslo,20200522,2020,vg.no,nob,None,None,None,None,nettavis
3,601822392,<urn:uuid:19ab616c-25bf-4bab-8dc1-e785c75f3701>,Østlendingen,None,https://www.ostlendingen.no:443/linda-onsker-r...,None,None,Elverum,20210611,2021,ostlendingen.no,nob,None,None,None,None,nettavis
4,601830429,<urn:uuid:68601964-c3a1-4beb-a725-952a0af5bbe1>,Ås Avis,None,https://www.aasavis.no:443/thea-17-og-rikke-17...,None,None,Ås,20220515,2022,aasavis.no,nob,None,None,None,None,nettavis
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59257,600818745,<urn:uuid:f3df54de-cbaa-48c4-9a00-68e8feb6cc41>,Forskning.no,None,https://forskning.no:443/etnisitet-bakgrunn-bi...,None,None,Oslo,20201110,2020,forskning.no,nob,None,None,None,None,nettavis
59258,600945978,<urn:uuid:1c73f501-751d-4259-bf77-38617f3a1b25>,Harstad Tidende,None,https://www.ht.no:443/nyheter/2020/11/04/Voksn...,None,None,Harstad,20201105,2020,ht.no,nob,None,None,None,None,nettavis
59259,600897303,<urn:uuid:c8e98047-67b0-4d4f-a922-c12b67d28da1>,Gausdølen,None,https://www.gausdolen.no:443/kultur-og-sport/j...,None,None,Gausdal,20200509,2020,gausdolen.no,nob,None,None,None,None,nettavis
59260,600816258,<urn:uuid:e9c89d9c-4516-4062-824a-6c08efaabf30>,Forskning.no,None,https://forskning.no:443/vold-partner-psykiske...,None,None,Oslo,20191111,2019,forskning.no,nob,None,None,None,None,nettavis


### 1.1.1 Dedupliser per URI

Selv om NB dedupliserer versjoner av nettsteder som er eksakt like, finnes det en hel del nesten-duplikater i tekstsamlingen. Disse er beholdt for å tillate en mengde ulike forskningsspørsmål, f.eks. analyse av svært like tekster.

I dette tilfelle ønsker vi å utelukke duplikater, og vi beholder derfor bare den tidligste versjonen av tekster som er hentet fra samme nettadresse.

In [15]:
corpus.frame = (
    corpus.frame
    .sort_values("timestamp")
    .drop_duplicates(subset="oaiid", keep="first")
    .reset_index(drop=True)
)

print(f"Deduplisering reduserte korpuset til {len(corpus.frame)} tekster.")

corpus

Deduplisering reduserte korpuset til 40178 tekster.


,dhlabid,urn,title,authors,oaiid,sesamid,isbn10,city,timestamp,year,publisher,langs,subjects,ddc,genres,literaryform,doctype
0,600214925,<urn:uuid:baece543-21c3-4ac7-a808-3bcb85b80d76>,Le Monde Diplomatique,None,https://www.lmd.no/lmd/verden/europa/,None,None,Oslo,20190218,2019,lmd.no,nob,None,None,None,None,nettavis
1,600772043,<urn:uuid:dd23225a-af6c-4674-9db7-b71bdcfc4124>,Fitjarposten,None,https://fitjarposten.no/category/nyhende/sport/,None,None,Fitjar,20190219,2019,fitjarposten.no,nno,None,None,None,None,nettavis
2,600768911,<urn:uuid:57143c8a-8aa3-47db-b58d-ab4b7700878d>,Fitjarposten,None,https://fitjarposten.no/category/nyhende/fitja...,None,None,Fitjar,20190219,2019,fitjarposten.no,nno,None,None,None,None,nettavis
3,601053591,<urn:uuid:56f5f336-c822-403e-967b-37cb70912ab0>,Karmsund avis,None,https://www.karmsundavis.no/kultur,None,None,Haugesund,20190219,2019,karmsundavis.no,nob,None,None,None,None,nettavis
4,600773707,<urn:uuid:78eda6b9-af3f-4161-9222-b4087b46ffb9>,Fitjarposten,None,https://fitjarposten.no/fitjar-idrettslag/,None,None,Fitjar,20190219,2019,fitjarposten.no,nno,None,None,None,None,nettavis
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40173,600896545,<urn:uuid:c4d6a8e1-ca18-4743-a9b5-271bb8e86f31>,Frostingen,None,https://www.frostingen.no:443/nyheter/na-avvik...,None,None,Frosta,20220619,2022,frostingen.no,nob,None,None,None,None,nettavis
40174,601421605,<urn:uuid:16eba9e8-72a4-4ed7-8ba2-14f2552f721d>,Romsdals Budstikke,None,https://www.rbnett.no:443/meninger/i/G3Vykq/br...,None,None,Molde,20220619,2022,rbnett.no,nob,None,None,None,None,nettavis
40175,601104258,<urn:uuid:7bc9e0f9-f87d-44b7-afee-c55bdbe852c2>,Kulingen,None,https://kulingen.no:443/lover-tidenes-konsert-...,None,None,Meløy,20220619,2022,kulingen.no,nob,None,None,None,None,nettavis
40176,601182860,<urn:uuid:37d111a3-8cfa-4cc8-959c-8b81b690750b>,Nordnorsk debatt,None,https://www.nordnorskdebatt.no:443/et-stort-so...,None,None,Tromsø,20220619,2022,nordnorskdebatt.no,nob,None,None,None,None,nettavis


#### OBS: Forbedre deduplisering

Dedupliseringen gjøres bare på eksakte URLer, men ikke der f.eks. parametre varierer for samme URL. Finne en måte å bare se på URL før `?`.

Eks på tekster som ikke dedupliseres:
601232685
601232694
601232688
601232697
601232687

### 1.2.1 Publikasjoner i korpus (tree map)

Vise distribusjon av tekster per publikasjon, og visualiserer fordelingen i et tree map.

In [ ]:
telle_titler = corpus['title'].value_counts()

fig = px.treemap(
    path=[telle_titler.index],
    values=telle_titler.values,
    title='Fordeling av tekster etter publikasjonsnavn',
    hover_data={'Antall tekster': telle_titler.values}
)

fig.show()


___
### Notat for pipeline

Vi utvikler pipeline basert på **NRK**.
___

Før du går videre og analyserer teksten ønsker du kanskje innsikt i korpuset du har bygget? La oss visualisere hvordan tekstene fordeler seg etter år, publikasjoner og språk.

### 1.2.2 Fordeling av tekster over tid

In [ ]:
hoestetid = pd.to_datetime(corpus['timestamp'], format='%Y%m%d')

# Generer tilfeldige verdier for spredning langs y-aksen
y_spredning = np.random.uniform(0, 1, size=len(hoestetid))

# Legg til jitter (tilfeldig justering langs y-aksen) på høstetidspunktene
x_jitter = hoestetid + pd.to_timedelta(np.random.uniform(-2, 2, size=len(hoestetid)), unit='D')

# Lager scatter plot med jitter for bedre spredning
fig = px.scatter(
    x=x_jitter,  # Høstetidspunkter med jitter på x-aksen
    y=y_spredning,  # Tilfeldig spredning på y-aksen
    labels={'x': 'Høstetidspunkt', 'y': 'Spredning'},
    title='Tekster etter høstedato',
    opacity=0.5,  # Gjør punktene delvis gjennomsiktige
    size_max=5  # Øk punktstørrelsen
)

# Vis diagrammet
fig.show()


### 1.2.3 Fordeling av språk/målform

In [ ]:
# Tell antall forekomster av hvert språk (langs-kode)
språkfordeling = corpus['langs'].value_counts()

# Lag et kakediagram basert på språkfordelingen
fig = px.pie(
    values=språkfordeling.values,  # Antall forekomster av hvert språk
    names=språkfordeling.index,    # Språkkoder (ISO 639-2)
    title='Fordeling av språk/målform',
    labels={'names': 'Språk/målform', 'values': 'Antall forekomster'}
)

# Vis diagrammet
fig.show()


## 1.3 Eksportere korpus

Hvis du vil lagre resultatene kan du eksportere dem i Excel og/eller JSONL format. Husk å endre filnavnet og ev. filsti!

In [ ]:
# Eksporter korpus til Excel (endre filnavn og filsti)
corpus.frame.to_excel('../export/korpus-ungdom-OR-ungdomm_.xlsx', index=False)

# Eksportere korpus til JSONL (endre filnavn og filsti)
# corpus.frame.to_json('../export/korpus-ungdom.jsonl', orient='records', lines=True)

___

## 2. Konkordanser

Når du har bygget korpuset kan du hente ut ulike informasjon om korpuset.

Kodecellen under henter ut konkordanser, det vil si et tekstvindu med inntil 12 ord før/etter et søkeord. Søkeordet fremheves i fet skrift.

La oss be om konkordanser for ordet "vindmølle".

In [4]:
conc_youth = corpus.conc(words="ungdom")
conc_youth.show()

,link,concordance
257,'>,Natur og Ungdom aksjonerer for å stanse Nordic Minings arbeid med å klargjøre tomten for gruvevirksomhet . Gruveselskapet har begynt...
102,'>,Dette skrek Lage Fossheim i bandet Brodways News til ungdom på 80-tallet . Den røde plastikkhansken som stupet røyksneiper brant...
130,'>,"Utlysningen er inspirert av prosjektet « Oppdrag 31 » , hvor ungdom fra hele landet deltok med innspill og kunnskap om hvordan..."
65,'>,"... har søkt , og fått innvilget åtte ukelønner på 7000 kroner hver , som de kan bruke på ungdom ."
62,'>,... Men aller viktigst er den hjelpen de har gitt til ungdom som sliter med å takle hverdagen .
234,'>,"Etter Gatelys aksjonen har Rørosingene fortsatt å engasjere seg , blant annet gjennom å være faste givere , “ Ungdom for..."
286,'>,Temagruppa for yngre ungdom : Konstituere gruppa med det første .
126,'>,"... omfatte barn 5 - 11 år utan alvorlege sjukdommar , samt dose 2 til ungdom 12 - 15 år ."
491,'>,"... 39 prosent av ungdom med innvandrerbakgrunn oppga å ha klart seg bra , mot bare 19 prosent av med norsk..."
160,'>,"... Avdelingssjefen ved Stovner politistasjon kalte Haugenstua for « en verkebyll i vår politikrets » , og politiet forbød ungdom å stå i..."


### Eksportere konkordanser

Hvis du vil lagre resultatene kan du eksportere dem i Excel og/eller JSONL format. Husk å endre filnavnet og ev. filsti!

In [ ]:
# Eksportere konkordanser til Excel
conc_youth.frame.to_excel('./konkordanser-vindmølle.xlsx', index=False)

# Eksportere konkordanser til JSONL
conc_youth.frame.to_json('./konkordanser-vindmølle.jsonl', orient='records', lines=True)

___

## 3. Kollokasjoner og ordfrekvenser
Kollokasjoner er ordpar som forekommer sammen. Ved å telle kollokasjoner for et gitt ord kan vi si noe om frekvensen for hvor hyppig/sjelden ulike ord forekommer sammen med hverandre.

En del ord alltid vil forekomme hyppig i alle tekster, slik som `og`, `han` og `hun`. For å finne ord som er signifikante innen en gitt kontekst kan vi beregne en relativfrekvens, der vi sammenlikner frekvensen i vårt eget korpus med et allment referansekorpus.

Cellen under lister ordene med høyest relativfrekvens, gitt et nøkkelord i korpuset.

In [20]:
tot = totals(75000) # Henter referansekorpus for å sammenlikne relativfrekvens
coll = corpus.coll("ungdom").frame.sort_values(by="counts", ascending=False) # Teller kollokasjoner for nøkkelord i korpuset
(coll.counts / tot.freq).sort_values(ascending = False).head(20) # Finner relativfrekvens i korpuset, sammenliknet med referansekorpuset

møteplasser           0.000838
sommerjobb            0.000736
helsestasjon          0.000714
cannabis              0.000692
pandemien             0.000672
fritidsaktiviteter    0.000619
aldersgruppen         0.000547
aldersgruppa          0.000501
vaksinering           0.000366
rusproblemer          0.000356
fritidstilbud         0.000342
smitte                0.000321
Tromsøya              0.000313
møteplass             0.000311
mindreårig            0.000308
Vennesla              0.000292
hendelse              0.000287
innvandrerbakgrunn    0.000286
helsefremmende        0.000280
muslimsk              0.000275
dtype: float64